# Data Preprocessing

EDA gave us a clear picture of what we're working with. Now the job is to make the raw text usable transforming it from messy, inconsistent strings into a clean, uniform representation that a vectoriser can work with reliably.

Models don't understand that "GREAT", "great", and "great!!" all carry the same meaning. They see three different tokens. Every inconsistency in the raw text is an opportunity for the model to learn a spurious pattern instead of genuine sentiment. Preprocessing is how we close those loopholes before they become problems.

The decisions made here as what to strip, what to keep, how to combine fields directly shape the feature space. We're not just cleaning data; we're making choices about what information the model is even allowed to see.

In [1]:
import pandas as pd

## Environment Setup

back to root to call functions from helpers.py file

In [2]:
import os
from pathlib import Path
print(Path.cwd())
os.chdir(Path('..').resolve())
from src.utils.helpers import clean_text, save
print(Path.cwd())

e:\AI_ML\proj\sentiment-analysis-of-amazon-reviews-using-machine-learning-ml-queens\notebooks
E:\AI_ML\proj\sentiment-analysis-of-amazon-reviews-using-machine-learning-ml-queens


## Loading the Balanced Training Data

We load the balanced dataset produced at the end of EDA — already cleaned of basic procedures. The `.info()` check below is a quick sanity check that everything still alright.

In [3]:
balanced_sample_train = pd.read_csv(r'data/processed/balanced_sample_train.csv', dtype=str, quoting=0)
balanced_sample_train.head()

,review_target,review_title,review_content,review_content_char_count,review_content_word_count
0,2,GREAT CAMRA,I HAVE HAD THE DX6340 FOR ABOUT A YEAR.I LOVE ...,586,108
1,1,not so great,I'm using this book in an introductory organic...,570,88
2,1,Inaccurate and disappointing,I only read the first few chapters and was bom...,214,40
3,1,Equus 3340,"Feels cheaply made, the battery contacts were ...",193,34
4,2,awesome sheets!,I love these sheets! They are sleek & smooth w...,198,38


In [4]:
balanced_sample_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 79972 entries, 0 to 79971
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   review_target              79972 non-null  str  
 1   review_title               79972 non-null  str  
 2   review_content             79972 non-null  str  
 3   review_content_char_count  79972 non-null  str  
 4   review_content_word_count  79972 non-null  str  
dtypes: str(5)
memory usage: 3.1 MB


## What We're Looking At

Three columns matter: `review_title`, `review_content`, and `review_target`. The target is straightforward 1 for negative, 2 for positive. The text columns are where the work is done.

Titles tend to be short, punchy, and deliberately expressive, customers often condense their entire opinion into a few words. Content is longer and more nuanced, but also noisier. Combining them gives the model access to both the headline sentiment and the full argument behind it, which is why we concatenate them rather than choosing one.

## Cleaning the Training Text

The cleaning pipeline does several things in sequence, and the order matters. We concatenate title and content *before* cleaning so the join character doesn't accidentally survive as an artefact then normalise to lowercase, and remove punctuation...

In [5]:
processed_train = balanced_sample_train.copy()
processed_train['review_content_cleaned'] = clean_text(processed_train['review_title'].fillna('') + ' ' + processed_train['review_content'].fillna(''))
processed_train.head()

,review_target,review_title,review_content,review_content_char_count,review_content_word_count,review_content_cleaned
0,2,GREAT CAMRA,I HAVE HAD THE DX6340 FOR ABOUT A YEAR.I LOVE ...,586,108,great camra dx6340 year love picture good 35m ...
1,1,not so great,I'm using this book in an introductory organic...,570,88,not great using book introductory organic spec...
2,1,Inaccurate and disappointing,I only read the first few chapters and was bom...,214,40,inaccurate disappointing read first chapter bo...
3,1,Equus 3340,"Feels cheaply made, the battery contacts were ...",193,34,equus 3340 feel cheaply made battery contact r...
4,2,awesome sheets!,I love these sheets! They are sleek & smooth w...,198,38,awesome sheet love sheet sleek smooth really c...


## What the Cleaned Text Looks Like

The `review_content_cleaned` column should now contain plain, lowercase text with no HTML, no punctuation, and no stray whitespace. Spot-checking a few rows is worth doing here, particularly any that looked unusual in the raw data (very short reviews, reviews with lots of special characters, non-English text that slipped through).

What we're looking for: the cleaned text should still be readable not near empty or too stripped

## Applying the Same Pipeline to Validation Data

The validation set must go through exactly the same cleaning steps as training, same function, same parameters, same concatenation logic. Any deviation creates a mismatch between the distribution the model was trained on and the distribution it's being evaluated against, which would make our validation metrics unreliable.

In [6]:
processed_valid = pd.read_csv(r'data/samples/sample_valid.csv', dtype=str, quoting=0)
processed_valid.head()

,review_target,review_title,review_content
0,2,Everything you need,This is a wonderful book. It may have been mea...
1,1,Important note about carrier,"The carrier is very cute, and lightweight...ho..."
2,1,Not a musical instrument -cannot be played,I bought (elsewhere) one of these harps for my...
3,2,Do I Iike this monitor? Well... I have 2!,I have 2 of these babies hooked up to a dual-o...
4,1,Very disappointing,This book is very poorly written and lacks of ...


In [7]:
processed_valid['review_content_cleaned'] = clean_text(processed_valid['review_title'].fillna('') + ' ' + processed_valid['review_content'].fillna(''))
processed_valid.head()


,review_target,review_title,review_content,review_content_cleaned
0,2,Everything you need,This is a wonderful book. It may have been mea...,everything need wonderful book may meant clerg...
1,1,Important note about carrier,"The carrier is very cute, and lightweight...ho...",important note carrier carrier very cute light...
2,1,Not a musical instrument -cannot be played,I bought (elsewhere) one of these harps for my...,not musical instrument cannot played bought el...
3,2,Do I Iike this monitor? Well... I have 2!,I have 2 of these babies hooked up to a dual-o...,iike monitor well baby hooked dual output digi...
4,1,Very disappointing,This book is very poorly written and lacks of ...,very disappointing book very poorly written la...


## Validation Data After Cleaning

The validation set is now in the same form as the training set: a single `review_content_cleaned` column containing lowercased, punctuation-free text. No information from the training set has leaked into this process.


## Applying the Pipeline to Test Data

The test set is treated with particular care. We don't examine its label distribution, don't compute statistics on it to inform any decisions, and we certainly don't adjust the cleaning pipeline based on anything we see in it. It's processed mechanically, exactly as training and validation were.

Note: unlike the training set (which concatenated title + content), the test cleaning here is applied separately to content and title. This gives us flexibility to experiment with different feature combinations at the modelling stage without needing to re-run preprocessing.

In [8]:
processed_test = pd.read_csv(r'data/samples/sample_test.csv', dtype=str, quoting=0)
processed_test.head()

,review_target,review_title,review_content
0,2,This is a great book,I must preface this by saying that I am not re...
1,1,Huge Disappointment.,"As a big time, long term Trevanian fan, I was ..."
2,2,Wayne is tight but cant hang with Turk.,This album is hot as it wants to be. However C...
3,2,Excellent,I read this book when I was in elementary scho...
4,1,Not about Anusara,Although this book is touted on several Anusar...


In [9]:
processed_test['review_content_cleaned'] = clean_text(processed_test['review_content'])
processed_test.head()

,review_target,review_title,review_content,review_content_cleaned
0,2,This is a great book,I must preface this by saying that I am not re...,must preface saying not religious but loved bo...
1,1,Huge Disappointment.,"As a big time, long term Trevanian fan, I was ...",big time long term trevanian fan extremely dis...
2,2,Wayne is tight but cant hang with Turk.,This album is hot as it wants to be. However C...,album hot want however cash money best album e...
3,2,Excellent,I read this book when I was in elementary scho...,read book elementary school probably fourth gr...
4,1,Not about Anusara,Although this book is touted on several Anusar...,although book touted several anusara web site ...


## Test Content After Cleaning

The test content column is now clean. The same observations apply as for training and validation — we'd expect the cleaned text to be readable, lowercase, and free of formatting noise. 

In [10]:
processed_test['review_title_cleaned'] = clean_text(processed_test['review_title'])
processed_test.head()

,review_target,review_title,review_content,review_content_cleaned,review_title_cleaned
0,2,This is a great book,I must preface this by saying that I am not re...,must preface saying not religious but loved bo...,great book
1,1,Huge Disappointment.,"As a big time, long term Trevanian fan, I was ...",big time long term trevanian fan extremely dis...,huge disappointment
2,2,Wayne is tight but cant hang with Turk.,This album is hot as it wants to be. However C...,album hot want however cash money best album e...,wayne tight but cannot hang turk
3,2,Excellent,I read this book when I was in elementary scho...,read book elementary school probably fourth gr...,excellent
4,1,Not about Anusara,Although this book is touted on several Anusar...,although book touted several anusara web site ...,not anusara


## Test Titles After Cleaning

Titles are cleaned separately and stored alongside content. This might seem like a small detail, but it reflects something we learned in EDA: titles and content carry different kinds of signal. Titles are often more sentiment-dense per word. Having them as a separate, clean column gives future modelling experiments the option to weight them differently or treat them as independent features.

Save the cleaned data

All three cleaned datasets are saved to `data/processed/`. From this point forward, every modelling notebook loads from here — no one touches the raw files again.


In [11]:
save(df_base='data/processed', df=processed_train, df_name='processed_train.csv')

Saved dataframe processed_train.csv to data\processed\processed_train.csv


{'csv': WindowsPath('data/processed/processed_train.csv')}

In [12]:
save(df_base='data/processed', df=processed_valid, df_name='processed_valid.csv')

Saved dataframe processed_valid.csv to data\processed\processed_valid.csv


{'csv': WindowsPath('data/processed/processed_valid.csv')}

In [13]:
save(df_base='data/processed', df=processed_test, df_name='processed_test.csv')

Saved dataframe processed_test.csv to data\processed\processed_test.csv


{'csv': WindowsPath('data/processed/processed_test.csv')}